In [90]:
import pandas as pd
import openpyxl
import numpy as np
import geopandas as gpd
#import fiona
import mercantile #to convert quadkey to geometry
from shapely.geometry import box
import time
import os

import matplotlib.pyplot as plt
import seaborn as sns
import yaml

with open("../config.yaml", "r") as file:
        config = yaml.safe_load(file)

output_dir = "../data/clean/"

config
#import fiona
#fiona.listlayers("ADMIN_EXPRESS.gpkg")


{'input_data': {'file1': '../data/raw/2024-10-01_performance_mobile_tiles.parquet',
  'file2': '../data/raw/ADE_4-0_GPKG_WGS84G_FRA-ED2026-01-19.gpkg',
  'file3': '../data/raw/fichier_diffusion_2024.xlsx',
  'file4': '../data/raw/2024-10-01_performance_fixed_tiles.parquet'},
 'output_data': {'file1': '../data/clean/file1_cleaned.csv',
  'file2': '../data/clean/file2_cleaned.csv'}}

In [91]:
#Read input files
#mobile_perf_df = pd.read_parquet(config['input_data']['file1'])
#mobile_perf_df = internet_perf_df.drop(columns = ['tile', 'tile_x', 'tile_y'])

internet_perf_df = pd.read_parquet(config['input_data']['file4'])
internet_perf_df = internet_perf_df.drop(columns = ['tile', 'tile_x', 'tile_y'])

#read rural zone file
rural_df = pd.read_excel(config['input_data']['file3'], sheet_name = "Maille communale")
rural_df = rural_df.drop(index = range(0,3))
rural_df.columns = rural_df.iloc[0]
rural_df = rural_df.drop(rural_df.index[0]).reset_index(drop=True)
internet_perf_df.head()

,quadkey,avg_d_kbps,avg_u_kbps,avg_lat_ms,avg_lat_down_ms,avg_lat_up_ms,tests,devices
0,0022133222312323,33682,3278,99,208.0,247.0,1,1
1,0022133222330031,97271,14686,192,400.0,386.0,2,2
2,0022133222330033,92047,27325,101,258.0,198.0,1,1
3,0022232311221201,537,255,1001,NaN,3555.0,1,1
4,0022302331120033,104525,4896,100,562.0,170.0,1,1


In [87]:
internet_perf_df.head()

,quadkey,avg_d_kbps,avg_u_kbps,avg_lat_ms,avg_lat_down_ms,avg_lat_up_ms,tests,devices
0,0022310222030300,16997,2239,744,1542.0,737.0,1,1
1,0022332203013331,4839,948,43,91.0,88.0,1,1
2,0022332203031110,169938,23880,45,516.0,439.0,3,2
3,0022332203031113,10550,15278,209,1025.0,1112.0,1,1
4,0022332203102232,4722,912,42,73.0,75.0,1,1


In [11]:
#Extract regions, depts and communes from GPKG file
regions = gpd.read_file(config['input_data']['file2'], layer = 'REGION')

depts = gpd.read_file(config['input_data']['file2'], layer = 'DEPARTEMENT')

communes = gpd.read_file(config['input_data']['file2'], layer = 'COMMUNE')


In [5]:
communes.code_insee.unique()

<ArrowStringArray>
['01001', '01002', '01004', '01005', '01006', '01031', '01007', '01008',
 '01009', '01010',
 ...
 '97605', '97608', '97607', '97614', '97602', '97603', '97601', '97606',
 '97609', '97617']
Length: 34877, dtype: str

In [92]:
#convert quadkeys to geometry
def quadkey_to_geom(qk):
    tile = mercantile.quadkey_to_tile(qk)
    bounds = mercantile.bounds(tile)
    
    return box(
        bounds.west,
        bounds.south,
        bounds.east,
        bounds.north
    )

#Apply function of my geopadas df: quakey to geometry and add geometry column
start_time = time.time()
print("Converting quadkeys to geometry in progress...")
internet_perf_df["geometry"] = internet_perf_df["quadkey"].apply(quadkey_to_geom)
end_time = time.time()
print(f"Converting quadkeys to geometry finished. Time taken: {end_time - start_time: .2f}seconds")

Converting quadkeys to geometry in progress...


In [15]:
#Convert pandas df to Geopandas DataFrame
def convert_df_to_gdf(df):
    """
    Input: pandas df
    convert pandas to geopandas df
    Outpu: geopandas df
    """
    start_time = time.time()
    print("Converting Pandas DF to GeoPandas DF in progress ...")
    df1 = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326") #EPSG:4326 = WGS84     
    end_time = time.time()
    print(f"Pandas df is converted to geopandas df. Time take is: {end_time - start_time: .2f}seconds")

    return df1

internet_perf_gdf = convert_df_to_gdf(internet_perf_df) #Do not change df name
display(internet_perf_gdf.head())
#print(internet_perf_gdf.crs)
#print(communes.crs)

Converting Pandas DF to GeoPandas DF in progress ...


Pandas df is converted to geopandas df. Time take is:  0.35seconds


,quadkey,avg_d_kbps,avg_u_kbps,avg_lat_ms,avg_lat_down_ms,avg_lat_up_ms,tests,devices,geometry
0,0022310222030300,16997,2239,744,1542.0,737.0,1,1,"POLYGON ((-163.00964 69.73904, -163.00964 69.7..."
1,0022332203013331,4839,948,43,91.0,88.0,1,1,"POLYGON ((-162.59766 66.89775, -162.59766 66.8..."
2,0022332203031110,169938,23880,45,516.0,439.0,3,2,"POLYGON ((-162.60315 66.89344, -162.60315 66.8..."
3,0022332203031113,10550,15278,209,1025.0,1112.0,1,1,"POLYGON ((-162.59766 66.89128, -162.59766 66.8..."
4,0022332203102232,4722,912,42,73.0,75.0,1,1,"POLYGON ((-162.58118 66.8956, -162.58118 66.89..."


In [17]:
# make sjoin for departments and regions and communes
def get_france_region_gdf(gdf): 
    """
    Input: geopandas df (performance gdf & admin gdf)
    join geopandas df with region gdf
    Outpu: geopandas df combined for FRance regions
    """
    
    start_time = time.time()
    
    print("Adding adminitrative regions to the data frame is ongoing ...")
    gdf0 = gpd.sjoin(
        gdf,
        regions,
        how = "inner",
        predicate = "intersects"
    )
    gdf0 = gdf0.drop(columns = "index_right")
    
    end_time = time.time()
    print(f"Adding adminitrative regions to the data frame is finished. Time taken is: {end_time - start_time: .2f}seconds")
    
    return gdf0
    
def add_dept_and_comm_to_perf_gdf(gdf1, gdf2):
    """
    Input: geopandas df (performance gdf & admin gdf)
    join geopandas df to geo to geopandas df
    Outpu: geopandas df combined
    """
    
    start_time = time.time()
    
    print("Adding adminitrative classes to the geo data frame is ongoing ...")
    gdf0 = gpd.sjoin(
        gdf1,
        gdf2,
        how = "left",
        predicate = "intersects"
    )
    gdf0 = gdf0.drop(columns = "index_right")
    
    end_time = time.time()
    print(f"Adding adminitrative classes to the geo data frame is finished. Time taken is: {end_time - start_time: .2f}seconds")
    
    return gdf0

def combine_reg_dept_comm(gdf):
    """
    combines the 4 gdf: internet_perf_gdf + regions + depts + communes
    """
    
    internet_perf_reg_fr_gdf =  get_france_region_gdf(gdf)
    internet_perf_reg_dept_gdf =  add_dept_and_comm_to_perf_gdf(internet_perf_reg_fr_gdf, depts)
    internet_perf_admin_gdf =  add_dept_and_comm_to_perf_gdf(internet_perf_reg_dept_gdf, communes)
    
    return internet_perf_admin_gdf


gdf_perf_admin_all = combine_reg_dept_comm(internet_perf_gdf)
gdf_perf_admin_all.head()

Adding adminitrative regions to the data frame is ongoing ...


Adding adminitrative regions to the data frame is finished. Time taken is:  588.58seconds
Adding adminitrative classes to the data frame is ongoing ...


Adding adminitrative classes to the data frame is finished. Time taken is:  111.82seconds
Adding adminitrative classes to the data frame is ongoing ...


Adding adminitrative classes to the data frame is finished. Time taken is:  6.43seconds


,quadkey,avg_d_kbps,avg_u_kbps,avg_lat_ms,avg_lat_down_ms,avg_lat_up_ms,tests,devices,geometry,cleabs_left,...,date_du_recensement,organisme_recenseur,code_insee_du_canton,code_insee_de_l_arrondissement,code_insee_du_departement,code_insee_de_la_region_right,codes_siren_des_epci,code_siren,code_postal,superficie_cadastrale
512007,0313133210211131,37463,3095,56,671.0,1659.0,2,1,"POLYGON ((-1.93359 49.71382, -1.93359 49.71738...",REGION__0000000000000028,...,2023-01-01,INSEE,5014,502,50,28,200067205,200065415,50440,14870
512008,0313133210300022,10464,6864,33,1320.0,3521.0,1,1,"POLYGON ((-1.9281 49.71027, -1.9281 49.71382, ...",REGION__0000000000000028,...,2023-01-01,INSEE,5014,502,50,28,200067205,200065415,50440,14870
512009,0313133210302311,160386,31957,26,473.0,1117.0,1,1,"POLYGON ((-1.88965 49.67829, -1.88965 49.68185...",REGION__0000000000000028,...,2023-01-01,INSEE,5014,502,50,28,200067205,200065415,50440,14870
512010,0313133210303210,147627,11420,36,593.0,516.0,1,1,"POLYGON ((-1.87317 49.67829, -1.87317 49.68185...",REGION__0000000000000028,...,2023-01-01,INSEE,5014,502,50,28,200067205,200065415,50440,14870
512011,0313133210303212,25397,492,54,804.0,4537.0,1,1,"POLYGON ((-1.87317 49.67474, -1.87317 49.67829...",REGION__0000000000000028,...,2023-01-01,INSEE,5014,502,50,28,200067205,200065415,50440,14870


In [22]:
#gdf_perf_admin_all.code_siren.unique()
#gdf_perf_admin_all.columns
gdf_perf_admin_all.isna().sum()
gdf_perf_admin_all.shape

(90035, 36)

In [21]:
# Renaming and selecting columns

def column_clean_up(gdf):
    """
    input = 
    Keep only france regions+depts+communes
    Rename the columns
    outpu: france_gdf
    """
    #gdf = combine_reg_dept_comm()
    selected_col = ['avg_d_kbps', 'avg_u_kbps', 'avg_lat_ms', 'avg_lat_down_ms',
               'avg_lat_up_ms', 'tests', 'devices','nom_officiel_en_majuscules_left',
               'code_insee_left', 'code_siren_left','nom_officiel_en_majuscules_right',
                'code_insee_right', 'nom_officiel_en_majuscules', 'statut',
               'code_insee', 'population', 'date_du_recensement',
               'code_postal', 'superficie_cadastrale'] #'quadkey',  'geometry',  

    new_col_names = ['avg_down_kbps', 'avg_up_kbps', 'avg_lat_ms', 'avg_lat_down_ms',
                'avg_lat_up_ms', 'nbr_tests', 'nbr_devices','region_name',
               'code_insee_region', 'code_siren_region','department_name',
                'code_insee_dept', 'commune_name', 'commune_status',
                'code_insee_comm', 'comm_population', 'date_du_recensement',
                'zip_code', 'superficie_cadastrale'
                ]

    gdf = gdf[selected_col] #reduce number of columns
    gdf.columns = new_col_names #rename columns
    #gdf = gdf.dropna(subset = ['department_name'])#.reset_index()
    return gdf
    

#gdf_perf_admin_all = combine_reg_dept_comm() #gdf with all countries

gdf_perf_admin_fr = column_clean_up(gdf_perf_admin_all) #gdf france
gdf_perf_admin_fr.to_csv(f"{output_dir}gdf_perf_admin_fr_inner.csv", index=False, encoding= "utf-8", sep = ";")


In [23]:
gdf_perf_admin_fr.info()

<class 'pandas.DataFrame'>
Index: 90035 entries, 512007 to 3361041
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   avg_down_kbps          90035 non-null  int64         
 1   avg_up_kbps            90035 non-null  int64         
 2   avg_lat_ms             90035 non-null  int64         
 3   avg_lat_down_ms        88076 non-null  float64       
 4   avg_lat_up_ms          89278 non-null  float64       
 5   nbr_tests              90035 non-null  int64         
 6   nbr_devices            90035 non-null  int64         
 7   region_name            90035 non-null  str           
 8   code_insee_region      90035 non-null  str           
 9   code_siren_region      90035 non-null  str           
 10  department_name        90035 non-null  str           
 11  code_insee_dept        90035 non-null  str           
 12  commune_name           90035 non-null  str           
 13  commune_st

In [85]:
# Convert the downmoad and upload speed fro kbps to mbps
def merge_with_rural_zone(gdf):
    #convert gdf to df
    gdf1 = gdf.copy()
    gdf1 = pd.DataFrame(gdf)
    gdf1 = pd.merge(gdf1, rural_df[["CODGEO","LIBDENS"]] , how = "left", left_on = "code_insee_comm", right_on = "CODGEO")
    gdf1 = gdf1.drop(columns = ["CODGEO","date_du_recensement"])
    return gdf1
internet_perf_df_fr= merge_with_rural_zone(gdf_perf_admin_fr)
internet_perf_df_fr.head()

,avg_down_kbps,avg_up_kbps,avg_lat_ms,avg_lat_down_ms,avg_lat_up_ms,nbr_tests,nbr_devices,region_name,code_insee_region,code_siren_region,department_name,code_insee_dept,commune_name,commune_status,code_insee_comm,comm_population,zip_code,superficie_cadastrale,LIBDENS
0,0.04,0.00,56,671.0,1659.0,2,1,NORMANDIE,28,200053403,MANCHE,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural
1,0.01,0.01,33,1320.0,3521.0,1,1,NORMANDIE,28,200053403,MANCHE,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural
2,0.16,0.03,26,473.0,1117.0,1,1,NORMANDIE,28,200053403,MANCHE,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural
3,0.15,0.01,36,593.0,516.0,1,1,NORMANDIE,28,200053403,MANCHE,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural
4,0.03,0.00,54,804.0,4537.0,1,1,NORMANDIE,28,200053403,MANCHE,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural


In [86]:
print(internet_perf_df_fr.avg_down_kbps.max())
print(internet_perf_df_fr["avg_up_kbps"].max())
internet_perf_df_fr.head()

1.5
0.81


,avg_down_kbps,avg_up_kbps,avg_lat_ms,avg_lat_down_ms,avg_lat_up_ms,nbr_tests,nbr_devices,region_name,code_insee_region,code_siren_region,department_name,code_insee_dept,commune_name,commune_status,code_insee_comm,comm_population,zip_code,superficie_cadastrale,LIBDENS
0,0.04,0.00,56,671.0,1659.0,2,1,NORMANDIE,28,200053403,MANCHE,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural
1,0.01,0.01,33,1320.0,3521.0,1,1,NORMANDIE,28,200053403,MANCHE,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural
2,0.16,0.03,26,473.0,1117.0,1,1,NORMANDIE,28,200053403,MANCHE,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural
3,0.15,0.01,36,593.0,516.0,1,1,NORMANDIE,28,200053403,MANCHE,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural
4,0.03,0.00,54,804.0,4537.0,1,1,NORMANDIE,28,200053403,MANCHE,50,LA HAGUE,Commune simple,50041,11484,50440,14870,Rural
